<a href="https://colab.research.google.com/github/COMP3608-Group-12/Project/blob/Dataset-1-Implementing-K-fold-for-Logistic-Regression/Group_12_Dataset_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#Initial test commit

## STEP 1: Importing Packages

In [ ]:
#importing packages
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn
import imblearn

## STEP 2: Load & Understand Data

In [ ]:
#MUST BE RUN inorder for file download to work
!pip install gdown

#Download dataset csv into runtime from google drive link
import gdown

url = "https://drive.google.com/uc?id=1mFcMnqHO11_g5Fw-t9TOQPWSj59WZ_t9"
gdown.download(url, "creditcard.csv", quiet=False)

#loading csv file into pandas dataframe
df = pd.read_csv('creditcard.csv')

Downloading...
From (original): https://drive.google.com/uc?id=1mFcMnqHO11_g5Fw-t9TOQPWSj59WZ_t9
From (redirected): https://drive.google.com/uc?id=1mFcMnqHO11_g5Fw-t9TOQPWSj59WZ_t9&confirm=t&uuid=41798b9d-950c-41c7-89a0-21af03607a6c
To: /content/creditcard.csv
100%|██████████| 151M/151M [00:01<00:00, 79.2MB/s]


In [ ]:
#printing dataset information (how many rows, columns etc)
print(df.info())
print('----------------------------------------------------------')

#checking if any values are missing
print(df.isnull().sum())
print('----------------------------------------------------------')

# Show ONLY columns with missing values
print("Columns with missing values:")
print(df.isnull().sum()[df.isnull().sum() > 0])
print('----------------------------------------------------------')

#Dropping rows with missing values
df = df.dropna()

#Verifying that those rows have been dropped and there are no missing values
print("Total missing values after dropping:")
print(df.isnull().sum().sum())  # should be 0
print('----------------------------------------------------------')

#previewing the data
print(df.head(5))
print('----------------------------------------------------------')

#checking how many transactions are fraudulent and how many are non-fraudulent
print(df['Class'].value_counts())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 284807 entries, 0 to 284806
Data columns (total 31 columns):
 #   Column  Non-Null Count   Dtype  
---  ------  --------------   -----  
 0   Time    284807 non-null  float64
 1   V1      284807 non-null  float64
 2   V2      284807 non-null  float64
 3   V3      284807 non-null  float64
 4   V4      284807 non-null  float64
 5   V5      284807 non-null  float64
 6   V6      284807 non-null  float64
 7   V7      284807 non-null  float64
 8   V8      284807 non-null  float64
 9   V9      284807 non-null  float64
 10  V10     284807 non-null  float64
 11  V11     284807 non-null  float64
 12  V12     284807 non-null  float64
 13  V13     284807 non-null  float64
 14  V14     284807 non-null  float64
 15  V15     284807 non-null  float64
 16  V16     284807 non-null  float64
 17  V17     284807 non-null  float64
 18  V18     284807 non-null  float64
 19  V19     284807 non-null  float64
 20  V20     284807 non-null  float64
 21  V21     28

## Step 3: Split Data into Features & Target

In [ ]:
#Splitting features and target
X = df.drop('Class', axis=1)
y = df['Class']

##Step 4: Decision Tree Model using KFold Cross Validation

In [ ]:
from sklearn.model_selection import StratifiedKFold
from sklearn.tree import DecisionTreeClassifier
from imblearn.over_sampling import SMOTE
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, classification_report, confusion_matrix

#Decision Tree Model
model_dt = DecisionTreeClassifier(random_state=42)

# StratifiedKFold
kf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

# Store results
all_results = []

# Manual KFold Loop
for fold, (train_index, test_index) in enumerate(kf.split(X, y), 1):

    print(f"\n================== Fold {fold} ==================")

    # Split
    X_train, X_test = X.iloc[train_index], X.iloc[test_index]
    y_train, y_test = y.iloc[train_index], y.iloc[test_index]

    # SMOTE on training data only
    smote = SMOTE(random_state=42)
    X_train_resampled, y_train_resampled = smote.fit_resample(X_train, y_train)

    # Train model
    model_dt.fit(X_train_resampled, y_train_resampled)

    # Predict on test data
    y_pred = model_dt.predict(X_test)
    y_prob = model_dt.predict_proba(X_test)[:, 1]

    #Performance metrics
    results = {
        'Accuracy': accuracy_score(y_test, y_pred),
        'Precision': precision_score(y_test, y_pred),
        'Recall': recall_score(y_test, y_pred),
        'F1 Score': f1_score(y_test, y_pred),
        'ROC-AUC': roc_auc_score(y_test, y_prob)
    }

    results_df = pd.DataFrame({
        "Metric": list(results.keys()),
        "Value": [round(value, 4) for value in results.values()]
    })

    print(results_df)

    print(f"\n============================================")
    print("Confusion Matrix:")
    print(confusion_matrix(y_test, y_pred))

    print(f"\n============================================")
    print("Classification Report:")
    print(classification_report(y_test, y_pred, digits=4))

    # Appending results
    all_results.append(results)

# Taking average of all results
results_df = pd.DataFrame(all_results)

avg_results = results_df.mean()
std_results = results_df.std()

final_df = pd.DataFrame({
    "Metric": avg_results.index,
    "Average Value": [round(val, 4) for val in avg_results.values],
    "Std Dev": [round(val, 4) for val in std_results.values]
})

print("\n================== AVERAGE PERFORMANCE ==================")
print(final_df)


================== Fold 1 ==================
      Metric   Value
0   Accuracy  0.9975
1  Precision  0.3814
2     Recall  0.7551
3   F1 Score  0.5068
4    ROC-AUC  0.8765

Confusion Matrix:
[[28372    60]
 [   12    37]]

Classification Report:
              precision    recall  f1-score   support

           0     0.9996    0.9979    0.9987     28432
           1     0.3814    0.7551    0.5068        49

    accuracy                         0.9975     28481
   macro avg     0.6905    0.8765    0.7528     28481
weighted avg     0.9985    0.9975    0.9979     28481


================== Fold 2 ==================
      Metric   Value
0   Accuracy  0.9981
1  Precision  0.4684
2     Recall  0.7551
3   F1 Score  0.5781
4    ROC-AUC  0.8768

Confusion Matrix:
[[28390    42]
 [   12    37]]

Classification Report:
              precision    recall  f1-score   support

           0     0.9996    0.9985    0.9990     28432
           1     0.4684    0.7551    0.5781        49

    accuracy     

## Step 6: Deep Neural Network using K-Fold Cross Validation

In [ ]:
#Runtime roughly around 14-15 minutes

#Importing relavent packages
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.layers import Input
from sklearn.model_selection import StratifiedKFold
from imblearn.over_sampling import SMOTE
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, classification_report, confusion_matrix
from sklearn.preprocessing import StandardScaler


#Creating DNN Model
def create_dnn_model(input_dim):
  model = models.Sequential([
    Input(shape = (input_dim,)),

    layers.Dense(32, activation = 'relu'),
    layers.BatchNormalization(),                                                    #Normalize outputs

    layers.Dense(16, activation = 'relu'),
    layers.Dropout(0.2),                                                           #Prevents overfitting (20% dropoff)

    layers.Dense(8, activation = 'relu'),

    layers.Dense(1, activation = 'sigmoid')
  ])

  model.compile(
      optimizer = tf.keras.optimizers.Adam(learning_rate = 0.001),
      loss = 'binary_crossentropy',
      metrics = [
          tf.keras.metrics.Precision(),
          tf.keras.metrics.Recall(),
          tf.keras.metrics.AUC()
      ]
  )
  return model


##K-Fold Setup, Training Model, Predicting and Metrics

#K-Fold setup
kf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
all_results = []

#Manual K-Fold loop
for fold, (train_index, test_index) in enumerate(kf.split(X, y), 1):
    print(f"\n================== Fold {fold} ==================")

    # Split
    X_train, X_test = X.iloc[train_index], X.iloc[test_index]
    y_train, y_test = y.iloc[train_index], y.iloc[test_index]

    #Scaling Data
    scaler = StandardScaler()

    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)


    # SMOTE on training data only
    smote = SMOTE(random_state=42)
    X_train_resampled, y_train_resampled = smote.fit_resample(X_train_scaled, y_train)


    #Training Model
    model = create_dnn_model(input_dim = X_train_resampled.shape[1])

    history = model.fit(
        X_train_resampled, y_train_resampled,
        epochs = 10,
        batch_size = 256,
        validation_split = 0.2,
        verbose = 1
    )

    #Predicting
    y_prob = model.predict(X_test_scaled, verbose = 1).flatten()
    y_pred = (y_prob > 0.5).astype(float)

    #Performance Metrics
    results = {
        'Accuracy': accuracy_score(y_test, y_pred),
        'Precision': precision_score(y_test, y_pred),
        'Recall': recall_score(y_test, y_pred),
        'F1 Score': f1_score(y_test, y_pred),
        'ROC-AUC': roc_auc_score(y_test, y_prob)
    }

    results_df_dnn_dataset1 = pd.DataFrame({"Metric": list(results.keys()), "Value": [round(value, 4) for value in results.values()]})
    print(results_df_dnn_dataset1)

    print("\n Confusion Matrix:")
    print(confusion_matrix(y_test, y_pred))

    print("\n Classification Report:")
    print(classification_report(y_test, y_pred, digits=4))


    #Appending all results
    all_results.append(results)


#Tabulated Average of Results
results_df_dnn_dataset1  = pd.DataFrame(all_results)
avg_results_dnn_dataset1 = results_df_dnn_dataset1.mean()
std_results_dnn_dataset1 = results_df_dnn_dataset1.std()
final_df_dnn_dataset1    = pd.DataFrame({"Metric": avg_results_dnn_dataset1.index,
                                         "Average Value": [round(val, 4) for val in avg_results_dnn_dataset1.values],
                                         "Std Dev": [round(val, 4) for val in std_results_dnn_dataset1.values],
                                         })

print("\n================== AVERAGE PERFORMANCE ==================")
print(final_df_dnn_dataset1.to_string(index=False))



================== Fold 1 ==================
Epoch 1/10
1600/1600 ━━━━━━━━━━━━━━━━━━━━ 10s 5ms/step - auc_14: 0.9962 - loss: 0.0757 - precision_14: 0.9640 - recall_14: 0.9627 - val_auc_14: 0.0000e+00 - val_loss: 0.0140 - val_precision_14: 1.0000 - val_recall_14: 0.9978
Epoch 2/10
1600/1600 ━━━━━━━━━━━━━━━━━━━━ 6s 4ms/step - auc_14: 0.9996 - loss: 0.0188 - precision_14: 0.9894 - recall_14: 0.9945 - val_auc_14: 0.0000e+00 - val_loss: 0.0061 - val_precision_14: 1.0000 - val_recall_14: 0.9999
Epoch 3/10
1600/1600 ━━━━━━━━━━━━━━━━━━━━ 8s 5ms/step - auc_14: 0.9997 - loss: 0.0128 - precision_14: 0.9927 - recall_14: 0.9971 - val_auc_14: 0.0000e+00 - val_loss: 0.0064 - val_precision_14: 1.0000 - val_recall_14: 0.9998
Epoch 4/10
1600/1600 ━━━━━━━━━━━━━━━━━━━━ 6s 4ms/step - auc_14: 0.9997 - loss: 0.0100 - precision_14: 0.9942 - recall_14: 0.9981 - val_auc_14: 0.0000e+00 - val_loss: 0.0024 - val_precision_14: 1.0000 - val_recall_14: 1.0000
Epoch 5/10
1600/1600 ━━━━━━━━━━━━━━━━━━━━ 7s 4ms/step - a

## Step 7: Logistic Regression with K Fold Cross Validation

In [6]:
from sklearn.model_selection import StratifiedKFold
from sklearn.linear_model import LogisticRegression
from imblearn.over_sampling import SMOTE
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, classification_report, confusion_matrix

#Logistic Regression Model
model_lr = LogisticRegression(max_iter=5000, class_weight='balanced', random_state=42)

# StratifiedKFold
kf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
